# Phase 6: RAG & Vector Databases
## Day 28: ChromaAndFAISS

Date: 2026-04-24

### Learning objectives
- Understand what a vector database does.
- Build a small local vector search index.
- Learn the ChromaDB collection pattern.
- Learn basic FAISS index types.
- Compare exact search and approximate search ideas.
- Practice similarity search with metadata.

In [ ]:
import os
import json
import math
import hashlib
import textwrap
from pprint import pprint

import numpy as np
import pandas as pd

try:
    import chromadb
    CHROMA_AVAILABLE = True
except Exception:
    chromadb = None
    CHROMA_AVAILABLE = False

try:
    import faiss
    FAISS_AVAILABLE = True
except Exception:
    faiss = None
    FAISS_AVAILABLE = False

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    SKLEARN_AVAILABLE = True
except Exception:
    TfidfVectorizer = None
    SKLEARN_AVAILABLE = False

def show(title, content):
    print("\n" + "=" * 82)
    print(title)
    print("=" * 82)
    print(textwrap.dedent(str(content)).strip())

np.set_printoptions(precision=4, suppress=True)

print("Setup complete.")
print("ChromaDB available:", CHROMA_AVAILABLE)
print("FAISS available:", FAISS_AVAILABLE)
print("scikit-learn available:", SKLEARN_AVAILABLE)

In [ ]:
chunks = [
    {
        "chunk_id": "CAM-001",
        "doc_id": "DOC-CAMPAIGN",
        "title": "Campaign Performance Notes",
        "text": "Spring Coffee Push was an Instagram campaign for Berlin customers with strong clicks and conversions."
    },
    {
        "chunk_id": "CAM-002",
        "doc_id": "DOC-CAMPAIGN",
        "title": "Campaign Performance Notes",
        "text": "Bank App Onboarding was an email campaign with low conversion and a generic subject line."
    },
    {
        "chunk_id": "CAM-003",
        "doc_id": "DOC-CAMPAIGN",
        "title": "Campaign Performance Notes",
        "text": "Yoga Studio Trial used TikTok short videos and performed well with beginner-friendly copy."
    },
    {
        "chunk_id": "OCR-001",
        "doc_id": "DOC-OCR",
        "title": "OCR Pipeline Notes",
        "text": "OpenCV preprocessing improves OCR quality with grayscale, denoising, thresholding, and deskewing."
    },
    {
        "chunk_id": "OCR-002",
        "doc_id": "DOC-OCR",
        "title": "OCR Pipeline Notes",
        "text": "Tesseract works well on clean scans while EasyOCR can help with natural images and multilingual text."
    },
    {
        "chunk_id": "OCR-003",
        "doc_id": "DOC-OCR",
        "title": "OCR Pipeline Notes",
        "text": "After OCR, raw text should be cleaned and converted into structured JSON with validation."
    },
    {
        "chunk_id": "RAG-001",
        "doc_id": "DOC-RAG",
        "title": "RAG Design Notes",
        "text": "A RAG system retrieves relevant chunks from a vector database before generating an answer."
    },
    {
        "chunk_id": "RAG-002",
        "doc_id": "DOC-RAG",
        "title": "RAG Design Notes",
        "text": "Chunking strategy affects retrieval quality, answer accuracy, and context length."
    },
    {
        "chunk_id": "RAG-003",
        "doc_id": "DOC-RAG",
        "title": "RAG Design Notes",
        "text": "Embeddings turn text chunks into vectors that can be searched with cosine similarity."
    },
]

chunk_df = pd.DataFrame(chunks)
chunk_df

## 1. What is a vector database?

A vector database stores embeddings and searches for similar vectors.

In RAG, it helps find the most relevant chunks for a user question.

In [ ]:
vector_db_jobs = pd.DataFrame([
    {"job": "Store vectors", "meaning": "Save embeddings for chunks or documents"},
    {"job": "Search vectors", "meaning": "Find nearest vectors for a query"},
    {"job": "Store metadata", "meaning": "Keep doc_id, title, page, and chunk_id"},
    {"job": "Filter results", "meaning": "Search only selected documents or categories"},
    {"job": "Return context", "meaning": "Send top chunks to the LLM"},
])

vector_db_jobs

In [ ]:
rag_flow = '''
User question
    ↓
Embed the question
    ↓
Search similar chunk vectors
    ↓
Return top chunks with metadata
    ↓
LLM answers using retrieved context
'''

print(rag_flow)

## 2. Create local embeddings

Real systems often use OpenAI embeddings or sentence-transformers.

For this notebook, we use TF-IDF or a hash fallback so everything runs locally.

In [ ]:
def simple_hash_embedding(text, dim=64):
    vector = np.zeros(dim, dtype=np.float32)

    for word in text.lower().split():
        digest = hashlib.md5(word.encode("utf-8")).hexdigest()
        index = int(digest[:8], 16) % dim
        sign = 1 if int(digest[8:16], 16) % 2 == 0 else -1
        vector[index] += sign

    norm = np.linalg.norm(vector)
    if norm == 0:
        return vector
    return vector / norm

texts = chunk_df["text"].tolist()

if SKLEARN_AVAILABLE:
    vectorizer = TfidfVectorizer(stop_words="english")
    embedding_matrix = vectorizer.fit_transform(texts).toarray().astype("float32")

    def embed_query(query):
        return vectorizer.transform([query]).toarray()[0].astype("float32")
else:
    embedding_matrix = np.vstack([simple_hash_embedding(text, dim=64) for text in texts]).astype("float32")

    def embed_query(query):
        return simple_hash_embedding(query, dim=64).astype("float32")

print("Embedding matrix shape:", embedding_matrix.shape)
print("First vector preview:", embedding_matrix[0][:10])

In [ ]:
def normalize_rows(matrix):
    matrix = np.array(matrix, dtype=np.float32)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return np.where(norms == 0, matrix, matrix / norms)

def normalize_vector(vector):
    vector = np.array(vector, dtype=np.float32)
    norm = np.linalg.norm(vector)
    return vector if norm == 0 else vector / norm

normalized_embeddings = normalize_rows(embedding_matrix)

print("First normalized vector length:", round(float(np.linalg.norm(normalized_embeddings[0])), 4))

## 3. Exact similarity search from scratch

Before using Chroma or FAISS, build the search manually.

This makes vector database behavior easier to understand.

In [ ]:
def cosine_similarity(a, b):
    a = np.array(a, dtype=np.float32)
    b = np.array(b, dtype=np.float32)

    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    if denominator == 0:
        return 0.0

    return float(np.dot(a, b) / denominator)

def exact_search(query, top_k=3, metadata_filter=None):
    query_vector = normalize_vector(embed_query(query))

    scores = normalized_embeddings @ query_vector

    results = chunk_df.copy()
    results["score"] = scores

    if metadata_filter:
        for key, value in metadata_filter.items():
            results = results[results[key] == value]

    return results.sort_values("score", ascending=False).head(top_k)

exact_search("Which campaign had a generic subject line?", top_k=3)

In [ ]:
queries = [
    "OCR preprocessing improves text extraction",
    "vector database retrieval for RAG",
    "TikTok campaign performed well",
]

for query in queries:
    print("\nQuery:", query)
    display(exact_search(query, top_k=2)[["chunk_id", "doc_id", "score", "text"]])

## 4. ChromaDB collection pattern

ChromaDB organizes vectors into collections.

A collection is like a table for documents, embeddings, and metadata.

In [ ]:
chroma_notes = '''
Install:
pip install chromadb

Common pattern:
client = chromadb.Client()
collection = client.get_or_create_collection("my_collection")

collection.add(
    ids=["chunk-1"],
    documents=["text goes here"],
    embeddings=[[0.1, 0.2, 0.3]],
    metadatas=[{"doc_id": "DOC-1"}]
)

collection.query(
    query_embeddings=[[0.1, 0.2, 0.3]],
    n_results=3
)
'''

print(chroma_notes)

In [ ]:
class SimpleChromaLikeCollection:
    def __init__(self, name):
        self.name = name
        self.ids = []
        self.documents = []
        self.embeddings = []
        self.metadatas = []

    def add(self, ids, documents, embeddings, metadatas):
        self.ids.extend(ids)
        self.documents.extend(documents)
        self.embeddings.extend([np.array(e, dtype=np.float32) for e in embeddings])
        self.metadatas.extend(metadatas)

    def query(self, query_embeddings, n_results=3, where=None):
        query_vector = normalize_vector(query_embeddings[0])
        matrix = normalize_rows(np.vstack(self.embeddings))
        scores = matrix @ query_vector

        rows = []
        for i, score in enumerate(scores):
            metadata = self.metadatas[i]
            if where:
                keep = all(metadata.get(k) == v for k, v in where.items())
                if not keep:
                    continue

            rows.append({
                "id": self.ids[i],
                "document": self.documents[i],
                "metadata": metadata,
                "distance": float(1 - score),
                "score": float(score)
            })

        rows = sorted(rows, key=lambda row: row["score"], reverse=True)[:n_results]

        return {
            "ids": [[row["id"] for row in rows]],
            "documents": [[row["document"] for row in rows]],
            "metadatas": [[row["metadata"] for row in rows]],
            "distances": [[row["distance"] for row in rows]],
            "scores": [[row["score"] for row in rows]]
        }

collection = SimpleChromaLikeCollection("study_chunks")

collection.add(
    ids=chunk_df["chunk_id"].tolist(),
    documents=chunk_df["text"].tolist(),
    embeddings=normalized_embeddings.tolist(),
    metadatas=chunk_df[["doc_id", "title"]].to_dict(orient="records")
)

print("Collection name:", collection.name)
print("Stored chunks:", len(collection.ids))

In [ ]:
query_vector = normalize_vector(embed_query("How does OCR preprocessing help?"))

chroma_like_result = collection.query(
    query_embeddings=[query_vector],
    n_results=3
)

pprint(chroma_like_result)

In [ ]:
filtered_result = collection.query(
    query_embeddings=[normalize_vector(embed_query("retrieval chunks vector database"))],
    n_results=3,
    where={"doc_id": "DOC-RAG"}
)

pprint(filtered_result)

## 5. Real ChromaDB optional path

If ChromaDB is installed, this cell creates a real in-memory collection.

If it is not installed, the notebook keeps using the simple fallback collection.

In [ ]:
def build_real_or_simple_chroma_collection(name, df, embeddings):
    if CHROMA_AVAILABLE:
        client = chromadb.Client()

        try:
            client.delete_collection(name)
        except Exception:
            pass

        real_collection = client.get_or_create_collection(name)
        real_collection.add(
            ids=df["chunk_id"].tolist(),
            documents=df["text"].tolist(),
            embeddings=embeddings.tolist(),
            metadatas=df[["doc_id", "title"]].to_dict(orient="records")
        )
        print("Using real ChromaDB collection.")
        return real_collection

    print("ChromaDB not installed. Using SimpleChromaLikeCollection.")
    simple_collection = SimpleChromaLikeCollection(name)
    simple_collection.add(
        ids=df["chunk_id"].tolist(),
        documents=df["text"].tolist(),
        embeddings=embeddings.tolist(),
        metadatas=df[["doc_id", "title"]].to_dict(orient="records")
    )
    return simple_collection

maybe_chroma_collection = build_real_or_simple_chroma_collection(
    "day28_chunks",
    chunk_df,
    normalized_embeddings
)

print(type(maybe_chroma_collection))

In [ ]:
def query_chroma_collection(collection, query, top_k=3):
    query_vector = normalize_vector(embed_query(query))

    result = collection.query(
        query_embeddings=[query_vector.tolist() if hasattr(query_vector, "tolist") else query_vector],
        n_results=top_k
    )

    return result

result = query_chroma_collection(maybe_chroma_collection, "generic email subject line", top_k=3)
pprint(result)

## 6. FAISS basics

FAISS is a fast vector similarity search library.

It focuses on indexes. You add vectors to an index, then search with query vectors.

In [ ]:
faiss_notes = '''
Install:
pip install faiss-cpu

Common index types:
- IndexFlatL2: exact L2 distance search
- IndexFlatIP: exact inner product search
- IndexIVFFlat: approximate search with clustering
- IndexHNSWFlat: graph-based approximate search

Basic pattern:
index = faiss.IndexFlatIP(dimension)
index.add(vectors)
scores, indices = index.search(query_vectors, top_k)
'''

print(faiss_notes)

In [ ]:
class SimpleFaissLikeIndex:
    def __init__(self, dimension, metric="ip"):
        self.dimension = dimension
        self.metric = metric
        self.vectors = None

    def add(self, vectors):
        vectors = np.array(vectors, dtype=np.float32)
        if vectors.shape[1] != self.dimension:
            raise ValueError("Vector dimension does not match index dimension.")

        if self.vectors is None:
            self.vectors = vectors
        else:
            self.vectors = np.vstack([self.vectors, vectors])

    def search(self, query_vectors, top_k):
        query_vectors = np.array(query_vectors, dtype=np.float32)

        if self.metric == "ip":
            scores = query_vectors @ self.vectors.T
            indices = np.argsort(-scores, axis=1)[:, :top_k]
            sorted_scores = np.take_along_axis(scores, indices, axis=1)
            return sorted_scores, indices

        distances = np.sum((query_vectors[:, None, :] - self.vectors[None, :, :]) ** 2, axis=2)
        indices = np.argsort(distances, axis=1)[:, :top_k]
        sorted_distances = np.take_along_axis(distances, indices, axis=1)
        return sorted_distances, indices

dimension = normalized_embeddings.shape[1]
simple_index = SimpleFaissLikeIndex(dimension=dimension, metric="ip")
simple_index.add(normalized_embeddings)

query_vector = normalize_vector(embed_query("RAG vector database retrieval")).reshape(1, -1)
scores, indices = simple_index.search(query_vector, top_k=3)

print("Scores:", scores)
print("Indices:", indices)
chunk_df.iloc[indices[0]]

## 7. Real FAISS optional path

If FAISS is installed, this cell creates a real `IndexFlatIP`.

If not, it uses the fallback index from above.

In [ ]:
def build_real_or_simple_faiss_index(vectors, metric="ip"):
    vectors = np.array(vectors, dtype=np.float32)
    dimension = vectors.shape[1]

    if FAISS_AVAILABLE:
        if metric == "ip":
            index = faiss.IndexFlatIP(dimension)
        else:
            index = faiss.IndexFlatL2(dimension)

        index.add(vectors)
        print("Using real FAISS index.")
        return index

    print("FAISS not installed. Using SimpleFaissLikeIndex.")
    index = SimpleFaissLikeIndex(dimension=dimension, metric=metric)
    index.add(vectors)
    return index

faiss_index = build_real_or_simple_faiss_index(normalized_embeddings, metric="ip")
print(type(faiss_index))

In [ ]:
def search_faiss_index(index, query, top_k=3):
    query_vector = normalize_vector(embed_query(query)).reshape(1, -1).astype("float32")
    scores, indices = index.search(query_vector, top_k)

    results = chunk_df.iloc[indices[0]].copy()
    results["score"] = scores[0]
    return results

search_faiss_index(faiss_index, "Which chunk explains thresholding and denoising?", top_k=3)

## 8. IndexFlatIP vs IndexFlatL2

Inner product works well when vectors are normalized.

L2 distance measures distance directly. Smaller is better.

In [ ]:
ip_index = build_real_or_simple_faiss_index(normalized_embeddings, metric="ip")
l2_index = build_real_or_simple_faiss_index(normalized_embeddings, metric="l2")

query = "email campaign low conversion"
query_vector = normalize_vector(embed_query(query)).reshape(1, -1).astype("float32")

ip_scores, ip_indices = ip_index.search(query_vector, 3)
l2_distances, l2_indices = l2_index.search(query_vector, 3)

print("Inner product top chunks:")
display(chunk_df.iloc[ip_indices[0]][["chunk_id", "text"]].assign(score=ip_scores[0]))

print("L2 top chunks:")
display(chunk_df.iloc[l2_indices[0]][["chunk_id", "text"]].assign(distance=l2_distances[0]))

## 9. Approximate search idea

Exact search compares the query with every vector.

Approximate search is faster on large datasets, but it may miss the true nearest result.

In [ ]:
index_types = pd.DataFrame([
    {
        "index_type": "Flat",
        "search_type": "Exact",
        "best_for": "Small to medium datasets or highest recall",
        "tradeoff": "Can be slow for huge datasets"
    },
    {
        "index_type": "IVF",
        "search_type": "Approximate",
        "best_for": "Large datasets",
        "tradeoff": "Needs training and tuning"
    },
    {
        "index_type": "HNSW",
        "search_type": "Approximate",
        "best_for": "Fast high-recall search",
        "tradeoff": "Uses more memory"
    },
])

index_types

In [ ]:
def recommend_faiss_index(num_vectors, need_highest_recall=True, memory_sensitive=False):
    if num_vectors < 100_000 and need_highest_recall:
        return "IndexFlatIP or IndexFlatL2"

    if memory_sensitive:
        return "IndexIVFFlat or compressed IVF variants"

    return "IndexHNSWFlat or IndexIVFFlat"

for num_vectors in [1_000, 100_000, 5_000_000]:
    print(num_vectors, "=>", recommend_faiss_index(num_vectors))

## 10. Metadata with FAISS

FAISS stores vectors, not rich metadata.

You usually keep metadata in a separate table and map FAISS indices back to rows.

In [ ]:
metadata_table = chunk_df[["chunk_id", "doc_id", "title", "text"]].copy()
metadata_table["vector_index"] = metadata_table.index

metadata_table.head()

In [ ]:
def search_faiss_with_metadata(index, query, top_k=3, doc_id_filter=None):
    raw_results = search_faiss_index(index, query, top_k=len(chunk_df))

    if doc_id_filter is not None:
        raw_results = raw_results[raw_results["doc_id"] == doc_id_filter]

    return raw_results.head(top_k)

search_faiss_with_metadata(
    faiss_index,
    "retrieval and vector database",
    top_k=3,
    doc_id_filter="DOC-RAG"
)

## 11. Compare exact, Chroma-like, and FAISS-like search

Different tools should return similar top results when they use the same vectors.

Small differences can happen because of distance metrics and normalization.

In [ ]:
comparison_query = "How do I improve OCR quality with preprocessing?"

exact_results = exact_search(comparison_query, top_k=3)[["chunk_id", "score"]]
faiss_results = search_faiss_index(faiss_index, comparison_query, top_k=3)[["chunk_id", "score"]]

chroma_result = collection.query(
    query_embeddings=[normalize_vector(embed_query(comparison_query))],
    n_results=3
)

chroma_results = pd.DataFrame({
    "chunk_id": chroma_result["ids"][0],
    "score": chroma_result["scores"][0]
})

print("Exact search:")
display(exact_results)

print("FAISS-like search:")
display(faiss_results)

print("Chroma-like search:")
display(chroma_results)

In [ ]:
def evaluate_search(search_function, test_cases, top_k=3):
    rows = []

    for case in test_cases:
        results = search_function(case["query"], top_k)
        retrieved = set(results["chunk_id"].tolist())
        expected = set(case["expected_chunk_ids"])
        hit = len(retrieved & expected) > 0

        rows.append({
            "query": case["query"],
            "expected": sorted(expected),
            "retrieved": results["chunk_id"].tolist(),
            "hit": hit
        })

    return pd.DataFrame(rows)

test_cases = [
    {"query": "generic subject line email campaign", "expected_chunk_ids": ["CAM-002"]},
    {"query": "OCR thresholding denoising deskewing", "expected_chunk_ids": ["OCR-001"]},
    {"query": "RAG retrieves chunks from vector database", "expected_chunk_ids": ["RAG-001"]},
]

exact_eval = evaluate_search(lambda q, k: exact_search(q, k), test_cases, top_k=3)
exact_eval

In [ ]:
print("Hit rate:", exact_eval["hit"].mean())

## Tricky bits

Vector databases are powerful, but they are not magic.

Most search problems come from poor embeddings, poor chunking, missing metadata, or no evaluation.

In [ ]:
tricky_bits = pd.DataFrame([
    {
        "problem": "Unnormalized vectors with inner product",
        "symptom": "Large-magnitude vectors dominate results",
        "fix": "Normalize vectors or use the right metric"
    },
    {
        "problem": "Missing metadata",
        "symptom": "You cannot cite or filter results",
        "fix": "Store metadata beside each vector"
    },
    {
        "problem": "Different embedding models",
        "symptom": "Query and document vectors do not match well",
        "fix": "Use the same model for indexing and querying"
    },
    {
        "problem": "No evaluation set",
        "symptom": "You cannot measure retrieval quality",
        "fix": "Create test queries with expected chunks"
    },
    {
        "problem": "Using approximate search too early",
        "symptom": "Relevant chunks are missed",
        "fix": "Start with exact search, then scale"
    },
])

tricky_bits

In [ ]:
def diagnose_vector_search_issue(symptom):
    symptom = symptom.lower()

    if "irrelevant" in symptom:
        return "Check embeddings, chunk quality, and metadata filters."
    if "cannot cite" in symptom:
        return "Metadata is missing or not returned with search results."
    if "slow" in symptom:
        return "Consider FAISS approximate indexes or filtering before search."
    if "missing relevant" in symptom:
        return "Check chunking, top_k, index type, and retrieval evaluation."
    return "Inspect embeddings, index metric, normalization, and test cases."

for symptom in [
    "Search returns irrelevant results",
    "The answer cannot cite sources",
    "Search is too slow",
    "Search is missing relevant chunks"
]:
    print(symptom, "=>", diagnose_vector_search_issue(symptom))

## Trick questions

1. Does FAISS store document text and metadata by itself?

<details>
<summary>Answer</summary>

No. FAISS stores vectors. You usually keep text and metadata in a separate table.

</details>

2. What is a ChromaDB collection?

<details>
<summary>Answer</summary>

A collection is a group of documents, embeddings, IDs, and metadata that can be queried together.

</details>

3. When is exact search useful?

<details>
<summary>Answer</summary>

It is useful for small to medium datasets or when you need the highest recall.

</details>

4. Why normalize vectors before inner product search?

<details>
<summary>Answer</summary>

Normalization makes inner product behave like cosine similarity.

</details>

5. Should you evaluate retrieval before building a chatbot?

<details>
<summary>Answer</summary>

Yes. The chatbot can only answer well if retrieval finds useful context.

</details>

## Exercises

Fill in each `___`. Run the cell to check your answer.

In [ ]:
# Exercise 1
# Normalize the embedding matrix row by row.

norm_matrix = ___

assert norm_matrix.shape == embedding_matrix.shape
assert abs(np.linalg.norm(norm_matrix[0]) - 1.0) < 1e-5
print("Exercise 1 passed.")

In [ ]:
# Exercise 2
# Run exact search for a RAG query.

results = ___

assert isinstance(results, pd.DataFrame)
assert len(results) == 3
assert "score" in results.columns
print("Exercise 2 passed.")

In [ ]:
# Exercise 3
# Create a Chroma-like collection and add chunks.

my_collection = SimpleChromaLikeCollection("exercise_collection")
my_collection.add(
    ids=___,
    documents=___,
    embeddings=___,
    metadatas=___
)

assert len(my_collection.ids) == len(chunk_df)
print("Exercise 3 passed.")

In [ ]:
# Exercise 4
# Query the Chroma-like collection.

query_vector = normalize_vector(embed_query("OCR preprocessing"))
query_result = ___

assert "ids" in query_result
assert len(query_result["ids"][0]) == 2
print("Exercise 4 passed.")

In [ ]:
# Exercise 5
# Build a FAISS-like index.

index = ___
index.add(normalized_embeddings)

assert index.dimension == normalized_embeddings.shape[1]
print("Exercise 5 passed.")

In [ ]:
# Exercise 6
# Search the FAISS-like index.

query_vector = normalize_vector(embed_query("email campaign conversion")).reshape(1, -1)
scores, indices = ___

assert scores.shape == (1, 3)
assert indices.shape == (1, 3)
print("Exercise 6 passed.")

In [ ]:
# Exercise 7
# Search FAISS with metadata filtering.

filtered = ___

assert isinstance(filtered, pd.DataFrame)
assert (filtered["doc_id"] == "DOC-RAG").all()
print("Exercise 7 passed.")

In [ ]:
# Exercise 8
# Evaluate exact search with test cases.

eval_df = ___

assert isinstance(eval_df, pd.DataFrame)
assert "hit" in eval_df.columns
print("Exercise 8 passed.")

## Solutions

<details>
<summary>Exercise 1 solution</summary>

```python
norm_matrix = normalize_rows(embedding_matrix)
```

</details>

<details>
<summary>Exercise 2 solution</summary>

```python
results = exact_search("RAG vector database retrieval", top_k=3)
```

</details>

<details>
<summary>Exercise 3 solution</summary>

```python
my_collection.add(
    ids=chunk_df["chunk_id"].tolist(),
    documents=chunk_df["text"].tolist(),
    embeddings=normalized_embeddings.tolist(),
    metadatas=chunk_df[["doc_id", "title"]].to_dict(orient="records")
)
```

</details>

<details>
<summary>Exercise 4 solution</summary>

```python
query_result = my_collection.query(
    query_embeddings=[query_vector],
    n_results=2
)
```

</details>

<details>
<summary>Exercise 5 solution</summary>

```python
index = SimpleFaissLikeIndex(dimension=normalized_embeddings.shape[1], metric="ip")
```

</details>

<details>
<summary>Exercise 6 solution</summary>

```python
scores, indices = index.search(query_vector, top_k=3)
```

</details>

<details>
<summary>Exercise 7 solution</summary>

```python
filtered = search_faiss_with_metadata(
    faiss_index,
    "retrieval and vector database",
    top_k=3,
    doc_id_filter="DOC-RAG"
)
```

</details>

<details>
<summary>Exercise 8 solution</summary>

```python
eval_df = evaluate_search(lambda q, k: exact_search(q, k), test_cases, top_k=3)
```

</details>

## Cumulative review exercises

These mix topics from Days 18 to 27. Fill in `___` and run each cell.

In [ ]:
# Review 1: Prompt engineering
# Choose the prompting style that uses examples.

prompt_style = ___

assert prompt_style.lower() == "few-shot"
print("Review 1 passed.")

In [ ]:
# Review 2: Structured output
# Parse JSON text.

json_text = '{"campaign": "Demo", "clicks": 100}'
parsed = ___

assert parsed["clicks"] == 100
print("Review 2 passed.")

In [ ]:
# Review 3: Information extraction
# Calculate conversion rate.

record = {"clicks": 1000, "conversions": 80}
conversion_rate = ___

assert abs(conversion_rate - 0.08) < 1e-9
print("Review 3 passed.")

In [ ]:
# Review 4: Tesseract basics
# Choose the English language code.

english_lang_code = ___

assert english_lang_code == "eng"
print("Review 4 passed.")

In [ ]:
# Review 5: EasyOCR
# Create a language list for English and German.

easyocr_languages = ___

assert easyocr_languages == ["en", "de"] or easyocr_languages == ["de", "en"]
print("Review 5 passed.")

In [ ]:
# Review 6: OpenCV preprocessing
# Choose the thresholding method useful for uneven lighting.

threshold_method = ___

assert threshold_method.lower() == "adaptive"
print("Review 6 passed.")

In [ ]:
# Review 7: OCR plus LLM pipeline
# Choose the structured output format.

structured_format = ___

assert structured_format.upper() == "JSON"
print("Review 7 passed.")

In [ ]:
# Review 8: Document intelligence
# Create a quality rule for receipts.

quality_rule = ___

assert "total" in quality_rule.lower() or "tax" in quality_rule.lower()
print("Review 8 passed.")

In [ ]:
# Review 9: Embeddings
# Calculate cosine similarity.

a = np.array([1, 0, 0])
b = np.array([1, 1, 0])
score = ___

assert 0.70 < score < 0.72
print("Review 9 passed.")

In [ ]:
# Review 10: Chunking
# Create overlapping chunks.

sample = "A B C D E F G H I J K L M N O P"
chunked = ___

assert isinstance(chunked, list)
assert len(chunked) > 1
print("Review 10 passed.")

## Cumulative review solutions

<details>
<summary>Show solutions</summary>

```python
# Review 1
prompt_style = "few-shot"

# Review 2
parsed = json.loads(json_text)

# Review 3
conversion_rate = record["conversions"] / record["clicks"]

# Review 4
english_lang_code = "eng"

# Review 5
easyocr_languages = ["en", "de"]

# Review 6
threshold_method = "adaptive"

# Review 7
structured_format = "JSON"

# Review 8
quality_rule = "Receipt item total plus tax should equal total."

# Review 9
score = cosine_similarity(a, b)

# Review 10
chunked = fixed_size_chunks_with_overlap(sample, chunk_size=8, overlap=2)
```

</details>

In [ ]:
cheat_sheet = '''
DAY 28 CHEAT SHEET: CHROMA AND FAISS

Vector database purpose:
- Store embeddings.
- Search similar vectors.
- Return document chunks with metadata.
- Support RAG retrieval.

ChromaDB pattern:
- Create client.
- Create or get collection.
- Add ids, documents, embeddings, metadata.
- Query with query embeddings.
- Useful when you want document storage plus metadata handling.

FAISS pattern:
- Create index.
- Add vectors.
- Search query vectors.
- Keep metadata in a separate table.
- Useful when you want fast vector search.

Common FAISS indexes:
- IndexFlatIP: exact inner product search.
- IndexFlatL2: exact L2 distance search.
- IndexIVFFlat: approximate search for large datasets.
- IndexHNSWFlat: graph-based approximate search.

Important rules:
- Use the same embedding model for docs and queries.
- Normalize vectors when using inner product as cosine similarity.
- Store chunk metadata.
- Evaluate retrieval with test queries.
- Start simple before adding approximate indexes.
'''

print(cheat_sheet)

## Next up: Day 29 — LangChainRetrievalQA

You will learn document loaders, chains, RetrievalQA, prompt templates, and memory.